In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score,KFold
from sklearn.metrics import roc_auc_score, log_loss
import os
import warnings
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBRegressor
warnings.filterwarnings("ignore",category = ConvergenceWarning)
os.chdir('E:/Machine_Learning/Cases/Concrete_Strength/')

In [3]:
df = pd.read_csv('Concrete_Data.csv')
df.head()

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [4]:
X,y = df.drop('Strength',axis=1),df['Strength']

In [7]:
kfold = KFold(n_splits=5,random_state=26,shuffle=True)
xgb = XGBRegressor()

In [9]:
result  = cross_val_score(xgb, X, y, cv=kfold)
result

array([0.92280108, 0.94498653, 0.92303601, 0.89233457, 0.955328  ])

In [11]:
from sklearn.model_selection import GridSearchCV
params = {
    'learning_rate' : np.linspace(0.01,0.9,10),
    'max_depth' : [None,2,3,4],
    'n_estimators' : [25,50,100]
}
gcv = GridSearchCV(xgb, param_grid=params, cv=kfold)
gcv.fit(X,y)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score :  0.935401012517735


## Training the model using best parameters and (data = Concrete_Data.csv) and testing it on testConcrete.csv

In [23]:
tst = pd.read_csv('testConcrete.csv')
tst

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210
5,150,23,23,114,4,883,638,36
6,480,64,0,292,3,896,776,180
7,393,49,82,132,1,887,830,271
8,284,63,1,138,1,804,725,44
9,206,38,0,103,2,818,719,191


In [28]:
## xbg_best = XGBRegressor(n_estimators = 100,learning_rate = 0.603, max_depth = 3)
xbg_best = XGBRegressor(**gcv.best_params_)
xbg_best.fit(X,y)
xbg_best.predict(tst)


array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

### Instead of training the best model separately, we can only write gcv.predict() and we will get same result
#### refit : bool  : default=True
#### It Refit an estimator using the best found parameters on the whole dataset.

In [25]:
gcv.predict(tst)

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)